In [1]:
!pip uninstall -y langchain langchain-core langchain-community langchain-classic langgraph langgraph-sdk langgraph-prebuilt groq langchain-groq httpx chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc opentelemetry-instrumentation
!pip install -q langchain==0.2.17 langchain-core==0.2.43 langchain-community==0.2.19 langchain-groq==0.1.9 groq==0.9.0 langchain-google-genai groq==0.9.0 langchain-groq==0.1.9 httpx==0.27.0 langchain-huggingface==0.0.3 chromadb==0.5.23 opentelemetry-api==1.27.0 opentelemetry-sdk==1.27.0


Found existing installation: langchain 1.3.1
Uninstalling langchain-1.3.1:
  Successfully uninstalled langchain-1.3.1
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
Found existing installation: langgraph 1.2.1
Uninstalling langgraph-1.2.1:
  Successfully uninstalled langgraph-1.2.1
Found existing installation: langgraph-sdk 0.3.14
Uninstalling langgraph-sdk-0.3.14:
  Successfully uninstalled langgraph-sdk-0.3.14
Found existing installation: langgraph-prebuilt 1.1.0
Uninstalling langgraph-prebuilt-1.1.0:
  Successfully uninstalled langgraph-prebuilt-1.1.0
Found existing installation: httpx 0.28.1
Uninstalling httpx-0.28.1:
  Successfully uninstalled httpx-0.28.1
Found existing installation: opentelemetry-api 1.38.0
Uninstalling opentelemetry-api-1.38.0:
  Successfully uninstalled opentelemetry-api-1.38.0
Found existing installation: opentelemetry-sdk 1.38.0
Uninstalling opentelemetry-sdk-1.38.0:
  Succ

In [1]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain.memory import ConversationBufferWindowMemory

In [ ]:
from getpass import getpass
from langchain_groq import ChatGroq
groq_key = getpass("Enter Groq API Key: ")

llm = ChatGroq(
    api_key=groq_key,
    model="llama-3.1-8b-instant",
    temperature=0
)



Enter Groq API Key: ··········


In [ ]:
SYS_PROMPT = """ Act as a helpful assistant and give brief answers"""
prompt = ChatPromptTemplate.from_messages([("system", SYS_PROMPT),
                                           MessagesPlaceholder(variable_name="history"),
                                           ("human", "{query}"),
                                           ])
memory = ConversationBufferWindowMemory(return_messages=True, k=2)

def get_memory_messages(query):
  return memory.load_memory_variables('query')['history']


conversion_chain=(RunnablePassthrough.assign(history=RunnableLambda(get_memory_messages))
                  | prompt
                  | llm)

In [ ]:
query={'query' : 'What are the first four colors of a rainbow?'}
response = conversion_chain.invoke(query)
memory.save_context(query, {"output": response.content})
print(response.content)

The first four colors of a rainbow, in order, are: 

1. Red
2. Orange
3. Yellow
4. Green


In [ ]:
query={'query' : 'and the other 3?'}
response = conversion_chain.invoke(query)
memory.save_context(query, {"output": response.content})
print(response.content)

The last three colors of a rainbow, in order, are:

1. Blue
2. Indigo
3. Violet


In [ ]:
query={'query' : 'Explain AI in 2 bullet points?'}
response = conversion_chain.invoke(query)
memory.save_context(query, {"output": response.content})
print(response.content)

Here are 2 bullet points explaining AI:

• **Artificial Intelligence (AI)**: AI refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and perception.
• **AI capabilities**: AI systems can analyze data, recognize patterns, and make predictions or recommendations, often using machine learning algorithms and natural language processing techniques to simulate human-like behavior.


In [ ]:
query={'query' : 'Now do the same for Deep Learning'}
response = conversion_chain.invoke(query)
memory.save_context(query, {"output": response.content})
print(response.content)

Here are 2 bullet points explaining Deep Learning:

• **Deep Learning**: Deep Learning is a subset of Machine Learning that involves the use of artificial neural networks with multiple layers to analyze and interpret data. These networks are designed to mimic the structure and function of the human brain.
• **Deep Learning capabilities**: Deep Learning algorithms can learn complex patterns and relationships in data, such as images, speech, and text, and can be used for tasks like image recognition, natural language processing, and predictive modeling.


In [ ]:
memory.load_memory_variables({})

{'history': [HumanMessage(content='Explain AI in 2 bullet points?'),
  AIMessage(content='Here are 2 bullet points explaining AI:\n\n• **Artificial Intelligence (AI)**: AI refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and perception.\n• **AI capabilities**: AI systems can analyze data, recognize patterns, and make predictions or recommendations, often using machine learning algorithms and natural language processing techniques to simulate human-like behavior.'),
  HumanMessage(content='Now do the same for Deep Learning'),
  AIMessage(content='Here are 2 bullet points explaining Deep Learning:\n\n• **Deep Learning**: Deep Learning is a subset of Machine Learning that involves the use of artificial neural networks with multiple layers to analyze and interpret data. These networks are designed to mimic the structure and function of the human brain.\n• **Deep Learning capab

In [ ]:
query={'query' : 'What have we discussed so far?'}
response = conversion_chain.invoke(query)
memory.save_context(query, {"output": response.content})
print(response.content)

We've discussed two topics:

1. Artificial Intelligence (AI)
2. Deep Learning


In [ ]:
memory.load_memory_variables({})

{'history': [HumanMessage(content='Now do the same for Deep Learning'),
  AIMessage(content='Here are 2 bullet points explaining Deep Learning:\n\n• **Deep Learning**: Deep Learning is a subset of Machine Learning that involves the use of artificial neural networks with multiple layers to analyze and interpret data. These networks are designed to mimic the structure and function of the human brain.\n• **Deep Learning capabilities**: Deep Learning algorithms can learn complex patterns and relationships in data, such as images, speech, and text, and can be used for tasks like image recognition, natural language processing, and predictive modeling.'),
  HumanMessage(content='What have we discussed so far?'),
  AIMessage(content="We've discussed two topics:\n\n1. Artificial Intelligence (AI)\n2. Deep Learning")]}

# Conversation Chains with ConversationSummaryMemory

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain.memory import ConversationSummaryMemory

In [ ]:
SYS_PROMPT = """ Act as a helpful assistant and give brief answers"""
prompt = ChatPromptTemplate.from_messages([("system", SYS_PROMPT),
                                           MessagesPlaceholder(variable_name="history"),
                                           ("human", "{query}"),
                                           ])
memory = ConversationSummaryMemory(return_messages=True, llm=llm)

def get_memory_messages(query):
  return memory.load_memory_variables('query')['history']


conversion_chain=(RunnablePassthrough.assign(history=RunnableLambda(get_memory_messages))
                  | prompt
                  | llm)

In [ ]:
query={'query' : 'Explain AI in 2 bullet points?'}
response = conversion_chain.invoke(query)
memory.save_context(query, {"output": response.content})
print(response.content)

Here are 2 brief bullet points explaining AI:

• **Artificial Intelligence (AI)**: AI refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and perception.

• **Key Characteristics**: AI systems use algorithms, data, and machine learning to simulate human-like behavior, enabling them to adapt, learn from experience, and improve their performance over time.


In [ ]:
query={'query' : 'Now do the same for Deep Learning'}
response = conversion_chain.invoke(query)
memory.save_context(query, {"output": response.content})
print(response.content)

Current summary:
The human asks for a brief explanation of artificial intelligence. The AI explains AI in 2 bullet points, stating that it refers to the development of computer systems that can perform tasks requiring human intelligence, and that AI systems use algorithms, data, and machine learning to simulate human-like behavior. The AI also mentions that it has numerous applications across various industries, including healthcare, finance, transportation, education, and customer service.

New lines of conversation:
Human: What is Deep Learning?
AI: Deep Learning is a subset of Machine Learning that involves the use of artificial neural networks with multiple layers to analyze and interpret complex data.

New summary:
The human asks for a brief explanation of artificial intelligence. The AI explains AI in 2 bullet points, stating that it refers to the development of computer systems that can perform tasks requiring human intelligence, and that AI systems use algorithms, data, and mac

In [ ]:
memory.load_memory_variables({})

{'history': [SystemMessage(content='Current summary:\nThe human asks for a brief explanation of artificial intelligence. The AI explains AI in 2 bullet points, stating that it refers to the development of computer systems that can perform tasks requiring human intelligence, and that AI systems use algorithms, data, and machine learning to simulate human-like behavior. The AI also mentions that it has numerous applications across various industries, including healthcare, finance, transportation, education, and customer service. Additionally, the AI explains that Deep Learning is a subset of Machine Learning that involves the use of artificial neural networks with multiple layers to analyze and interpret complex data.\n\nNew lines of conversation:\nHuman: What are some real-world examples of Deep Learning applications?\nAI: Deep Learning has been successfully applied in image and speech recognition, natural language processing, and self-driving cars.\n\nNew summary:\nThe human asks for a

In [ ]:
query={'query' : 'What have we discussed so far?'}
response = conversion_chain.invoke(query)
memory.save_context(query, {"output": response.content})
print(response.content)

We've discussed the topic of Artificial Intelligence (AI). I provided a brief explanation of AI in 2 bullet points, and then we talked about Deep Learning, a subset of Machine Learning. I also gave some real-world examples of Deep Learning applications.


# Conversation Chains with VectorStoreRetrieverMemory

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain.memory import VectorStoreRetrieverMemory

In [ ]:
!pip install -q sentence-transformers protobuf==5.29.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 9.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.6 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
opentelemetry-proto 1.27.0 requires protobuf<5.0,>=3.19, but you have protobuf 5.29.5 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.27.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.27.0 which is incompatible.
google-adk 1.29.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-co

In [ ]:
from sentence_transformers import SentenceTransformer

class SentenceTransformerEmbeddings:
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name)

    def embed_query(self, text):
        return self.model.encode(text).tolist()

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

embed_model = SentenceTransformerEmbeddings(
    "BAAI/bge-small-en-v1.5"
)

In [ ]:
from langchain_community.vectorstores import Chroma

chroma_db = Chroma(
    collection_name="history_db",
    embedding_function=embed_model
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
SYS_PROMPT = """ Act as a helpful assistant and give brief answers"""
prompt = ChatPromptTemplate.from_messages([("system", SYS_PROMPT),
                                           MessagesPlaceholder(variable_name="history"),
                                           ("human", "{query}"),
                                           ])

retriever=chroma_db.as_retriever(search_type="similarity", search_kwargs={"k":2})
memory = VectorStoreRetrieverMemory(return_messages=True, retriever=retriever,input_key="query",
    memory_key="history")

def get_memory_messages(inputs):
    return memory.load_memory_variables(inputs)["history"]


conversion_chain=(RunnablePassthrough.assign(history=RunnableLambda(get_memory_messages))
                  | prompt
                  | llm)

In [ ]:
query={'query' : 'Tell me about AI'}
response = conversion_chain.invoke(query)
memory.save_context(query, {"output": response.content})
print(response.content)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


ValueError: variable history should be a list of base messages, got  of type <class 'str'>

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain.memory import VectorStoreRetrieverMemory

SYS_PROMPT = """Act as a helpful assistant and give brief answers."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYS_PROMPT),
    ("human", """
Relevant previous context:
{history}

Current question:
{query}
""")
])

retriever = chroma_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

memory = VectorStoreRetrieverMemory(
    retriever=retriever,
    input_key="query",
    memory_key="history",
    return_docs=False
)

def get_memory_text(inputs):
    return memory.load_memory_variables(inputs)["history"]

conversation_chain = (
    RunnablePassthrough.assign(
        history=RunnableLambda(get_memory_text)
    )
    | prompt
    | llm
)

query = {"query": "Tell me about AI"}

response = conversation_chain.invoke(query)

memory.save_context(
    {"query": query["query"]},
    {"output": response.content}
)

print(response.content)

Artificial Intelligence (AI) is a broad field of computer science that focuses on creating intelligent machines capable of performing tasks that typically require human intelligence, such as:

1. **Learning**: AI systems can learn from data and improve their performance over time.
2. **Reasoning**: AI can make decisions based on logic, rules, and patterns.
3. **Problem-solving**: AI can solve complex problems and optimize processes.
4. **Perception**: AI can interpret and understand data from sensors, images, and speech.

AI has many applications, including:

1. **Virtual assistants**: Siri, Alexa, and Google Assistant.
2. **Image recognition**: Self-driving cars, facial recognition, and medical diagnosis.
3. **Natural language processing**: Chatbots, language translation, and text analysis.
4. **Predictive analytics**: Forecasting sales, weather, and stock prices.

There are several types of AI, including:

1. **Narrow or weak AI**: Designed to perform a specific task, like playing ch

In [ ]:
query = {"query": "What did I ask earlier?"}

response = conversation_chain.invoke(query)

memory.save_context(
    {"query": query["query"]},
    {"output": response.content}
)

print(response.content)

You asked me to tell you about Artificial Intelligence (AI).


In [ ]:
query = {"query": "What about Deep Learning"}

response = conversation_chain.invoke(query)

memory.save_context(
    {"query": query["query"]},
    {"output": response.content}
)

print(response.content)

Deep Learning is a subset of Machine Learning, which is a part of Artificial Intelligence (AI). It's a type of neural network that's inspired by the structure and function of the human brain. Deep Learning is particularly good at:

1. **Image recognition**: Identifying objects, scenes, and actions in images.
2. **Natural language processing**: Understanding and generating human language.
3. **Speech recognition**: Transcribing spoken words into text.

Deep Learning is used in many applications, including:

1. **Self-driving cars**: Recognizing objects and making decisions.
2. **Virtual assistants**: Understanding voice commands.
3. **Image classification**: Identifying objects in images.
4. **Chatbots**: Understanding and responding to user queries.

Deep Learning is known for its ability to learn complex patterns in data, making it a powerful tool for many AI applications.


# Multi-user Conversation Chains with ChatMessageHistory

In [3]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain.memory import VectorStoreRetrieverMemory

In [4]:
history_store={}
def get_session_history(session_id):
  if session_id not in history_store:
    history_store[session_id]=ChatMessageHistory()
  return history_store[session_id]


SYS_PROMPT = """ Act as a helpful AI assistant"""
prompt_template = ChatPromptTemplate.from_messages([("system", SYS_PROMPT),
                                           MessagesPlaceholder(variable_name="history"),
                                           ("human", "{human_input}"),
                                           ])
llm_chain=(prompt_template|llm)

conv_chain=RunnableWithMessageHistory(llm_chain,get_session_history, input_messages_key="human_input", history_messages_key="history")

def chat_with_llm(prompt: str, session_id: str):
  for chunk in conv_chain.stream({"human_input": prompt}, {'configurable' : {'session_id': session_id}}):
    print(chunk.content, end = "")

Test Conversation chain for user 1

In [5]:
user_id = 'bob123'
prompt = "Hi I am Bob, can you explain AI in 2 bullet points?"
chat_with_llm(prompt, user_id)

Hi Bob, nice to meet you. I'd be happy to explain AI in simple terms. Here are 2 bullet points to get you started:

• **What is AI?**: Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and perception. AI systems can analyze data, recognize patterns, and make predictions or recommendations based on that data.

• **How does AI work?**: AI works by using algorithms and machine learning techniques to process and analyze large amounts of data. These algorithms can be trained on data to learn patterns and relationships, and then use that knowledge to make predictions or take actions. Think of it like a super-smart computer that can learn from experience and improve its performance over time.

Let me know if you have any questions or if you'd like me to explain more!

In [6]:
prompt = "Now do the same for Reinforcement learning"
chat_with_llm(prompt, user_id)

Hi Bob, I'd be happy to explain Reinforcement Learning in simple terms. Here are 2 bullet points to get you started:

• **What is Reinforcement Learning?**: Reinforcement Learning (RL) is a type of machine learning where an AI agent learns to take actions in an environment to maximize a reward or minimize a penalty. The agent learns through trial and error, receiving feedback in the form of rewards or penalties for its actions. The goal is to learn a policy that maps states to actions that lead to the highest cumulative reward.

• **How does Reinforcement Learning work?**: In RL, the agent interacts with the environment by taking actions and observing the consequences of those actions. The agent uses this feedback to update its policy, which is a set of rules that determine the actions to take in different situations. The agent learns to balance exploration (trying new actions) and exploitation (choosing actions that are likely to lead to a high reward). Think of it like a child learni

In [7]:
prompt = "Discuss briefly what we have discussed so far in bullet points?"
chat_with_llm(prompt, user_id)

Here's a brief summary of what we've discussed so far:

• **Artificial Intelligence (AI)**: I explained AI in 2 bullet points, covering its definition and how it works.
• **Reinforcement Learning (RL)**: I explained RL in 2 bullet points, covering its definition and how it works.

Let me know if you'd like to discuss more topics or if you have any specific questions!

Now Test Conversation chain for user 2

In [8]:
user_id = 'james007'
prompt = "Hi can you explain LLM in 2 bullet points?"
chat_with_llm(prompt, user_id)

Here are 2 bullet points explaining Large Language Models (LLMs):

• **What is an LLM?**: A Large Language Model (LLM) is a type of artificial intelligence (AI) model that is trained on vast amounts of text data to generate human-like language. These models use complex algorithms to analyze and understand the patterns, relationships, and context within the text, allowing them to generate coherent and context-specific responses.

• **How does an LLM work?**: LLMs work by taking in a prompt or input, processing it through multiple layers of neural networks, and generating a response based on the patterns and relationships learned from the training data. This process is often referred to as "predictive text generation," where the model predicts the next word or phrase in a sequence based on the context and patterns it has learned.

In [9]:
prompt = "Now do the same for Deep learning"
chat_with_llm(prompt, user_id)

Here are 2 bullet points explaining Deep Learning:

• **What is Deep Learning?**: Deep Learning is a subset of Machine Learning that involves the use of artificial neural networks with multiple layers to analyze and interpret data. These neural networks are designed to mimic the structure and function of the human brain, with each layer processing and transforming the input data in a hierarchical manner. This allows Deep Learning models to learn complex patterns and relationships in data, enabling them to perform tasks such as image and speech recognition, natural language processing, and more.

• **How does Deep Learning work?**: Deep Learning works by using a process called backpropagation to adjust the weights and biases of the neural network layers during training. This involves feeding the input data through the network, calculating the error between the predicted output and the actual output, and then adjusting the weights and biases to minimize the error. This process is repeate

In [10]:
prompt = "Discuss briefly what we have discussed so far in bullet points?"
chat_with_llm(prompt, user_id)

Here's a brief summary of our discussion so far:

• **Large Language Models (LLMs)**: We discussed what LLMs are and how they work, including their ability to generate human-like language and analyze complex patterns in text data.
• **Deep Learning**: We explained what Deep Learning is and how it works, including its use of artificial neural networks with multiple layers to analyze and interpret data, and the process of backpropagation to adjust the weights and biases of the network during training.

# Multi-user Window-based Conversation Chains with persistence- SQLChatMessageHistory

In [11]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain.memory import VectorStoreRetrieverMemory

def get_session_history_db(session_id):
  return SQLChatMessageHistory(session_id, "sqlite:///memory.db")


SYS_PROMPT = """ Act as a helpful AI assistant"""
prompt_template = ChatPromptTemplate.from_messages([("system", SYS_PROMPT),
                                           MessagesPlaceholder(variable_name="history"),
                                           ("human", "{human_input}"),
                                           ])
def memory_buffer_window(messages, k=2):
  return messages[-(k+1):]


llm_chain=(RunnablePassthrough.assign(history=lambda x: memory_buffer_window(x["history"]))
    | prompt_template| llm)



In [12]:
conv_chain=RunnableWithMessageHistory(llm_chain,get_session_history_db, input_messages_key="human_input",
history_messages_key="history")

def chat_with_llm(prompt: str, session_id: str):
  for chunk in conv_chain.stream({"human_input": prompt}, {'configurable' : {'session_id': session_id}}):
    print(chunk.content, end = "")

Test Conversation chain for user 1

In [13]:
user_id = 'Ram123'
prompt = "Hi I am Ram, can you explain horoscope in 2 bullet points?"
chat_with_llm(prompt, user_id)

/usr/local/lib/python3.12/dist-packages/langchain_core/runnables/history.py:567: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  message_history = self.get_session_history(


Hello Ram, I'd be happy to explain horoscopes in 2 bullet points:

* **What is a Horoscope?**: A horoscope is a chart or diagram that represents the positions of celestial bodies, such as planets and stars, at a specific time and place. It's used to gain insights into a person's personality, behavior, and future events based on their birth date, time, and location.
* **How Horoscopes Work**: Horoscopes are based on the idea that the positions of celestial bodies influence human affairs. By analyzing the positions of planets and other celestial bodies, astrologers can make predictions about a person's life, including their strengths, weaknesses, and potential challenges. There are 12 zodiac signs, each associated with a specific period of the year, and people are classified into one of these signs based on their birth date.

Which zodiac sign are you, Ram?

In [14]:
prompt = "What about Largest animal"
chat_with_llm(prompt, user_id)

The largest animal that has ever existed is the **Blue Whale**. On average, an adult blue whale can grow up to 82 feet (25 meters) in length and weigh around 150-170 tons (136,000-152,000 kilograms). However, the largest blue whale ever recorded was a female that was found in 1947 off the coast of Iceland, which measured around 108 feet (33 meters) in length and weighed an estimated 210 tons (182,000 kilograms).

Here are some interesting facts about blue whales:

* They are not only the largest animals on Earth but also the largest animals to have ever existed on our planet.
* Blue whales can live for up to 90 years in the wild.
* They are filter feeders, using their baleen plates to strain tiny krill and other organisms from the water.
* Blue whales can produce sounds that are loud enough to be heard for hundreds of miles.

Isn't that amazing, Ram?

In [15]:
prompt = "Discuss briefly what we have discussed so far in bullet points?"
chat_with_llm(prompt, user_id)

Here's a brief summary of what we've discussed so far:

* **Horoscopes**: We discussed what a horoscope is and how it works, including the idea that the positions of celestial bodies influence human affairs.
* **Largest Animal**: We talked about the blue whale, which is the largest animal that has ever existed, and shared some interesting facts about it, such as its massive size, lifespan, and feeding habits.

Let me know if you'd like to discuss anything else, Ram!

Test Conversation chain for user 2

In [16]:
user_id = 'Laxman123'
prompt = "Hi I am Laxman, can you  horoscope actually predict future in 2 bullet points?"
chat_with_llm(prompt, user_id)

Hello Laxman, nice to meet you. I'd be happy to help you understand horoscopes. Here are 2 bullet points that summarize the concept of horoscopes:

* **Horoscopes are based on astrology**: Horoscopes are based on the positions of stars and planets at the time of a person's birth. Astrologers believe that the positions of these celestial bodies influence a person's personality, behavior, and future events. By analyzing these positions, horoscopes aim to provide insights into a person's life, including their strengths, weaknesses, and potential challenges.
* **Horoscopes are not scientifically proven to predict the future**: While horoscopes can be entertaining and thought-provoking, there is no scientific evidence to support the idea that they can accurately predict the future. Many experts consider horoscopes to be a form of entertainment or a way to inspire self-reflection, rather than a reliable tool for predicting future events.

It's worth noting that horoscopes are often based on 

In [17]:
prompt = "shoudld one believe in horscope?"
chat_with_llm(prompt, user_id)

Laxman, whether or not to believe in horoscopes is a personal decision that depends on your individual perspective and values. Here are some points to consider:

**Reasons to believe in horoscopes:**

1. **Entertainment value**: Horoscopes can be a fun and engaging way to explore your personality, interests, and potential challenges.
2. **Self-reflection**: Horoscopes can inspire self-reflection and introspection, helping you to better understand yourself and your place in the world.
3. **Cultural significance**: Horoscopes have been a part of many cultures for centuries, and can be a way to connect with your heritage and traditions.
4. **Inspiration**: Horoscopes can provide inspiration and guidance, helping you to make decisions and navigate life's challenges.

**Reasons not to believe in horoscopes:**

1. **Lack of scientific evidence**: There is no scientific evidence to support the idea that horoscopes can accurately predict the future or influence a person's life.
2. **Generalize

In [18]:
prompt = "Discuss briefly what we have discussed so far in bullet points?"
chat_with_llm(prompt, user_id)

Here's a brief summary of our discussion so far:

* **Horoscopes explained**: I explained that horoscopes are based on astrology and are not scientifically proven to predict the future.
* **Should one believe in horoscopes?**: We discussed the pros and cons of believing in horoscopes, including their entertainment value, self-reflection, cultural significance, and inspiration, as well as their lack of scientific evidence, generalized predictions, superstition, and unrealistic expectations.
* **Personal decision**: We concluded that whether or not to believe in horoscopes is a personal decision that depends on individual perspectives and values.

Let me know if you'd like to discuss anything else, Laxman!